# Case pipeline_lab — Churn (Databricks)

Funil completo via **`Esteira`** (`modelagem_lab.Esteira`, a partir da v0.2.0): um builder encadeável que guarda `df_dev`/`df_teste` e os artefatos de cada etapa como estado, em vez de você desempacotar manualmente o retorno de cada função solta (`divisao → construcao → agregacao_temporal → interacao → categorizar → preselecao → treinamento`).

Cada chamada mostra **todos os parâmetros**, mesmo os que ficam no valor default — assim serve de referência do que dá pra ajustar. Ver `python/pipeline_lab/REFERENCIA.md` (ou o PDF em `docs/pipeline_lab_referencia.pdf`) para a explicação de cada parâmetro e a literatura por trás, e a docstring de `pipeline_lab` pro estilo funcional equivalente (cada etapa como função solta, sem o builder).

## 0. Instalação

In [ ]:
%pip install --quiet https://github.com/PedroMaiorano/modelagem-lab/releases/download/v0.2.2/modelagem_lab-0.2.2-py3-none-any.whl

In [ ]:
dbutils.library.restartPython()

## 1. Carregar os dados (Spark → pandas)

`sample_submission`/`test` são a base sem rótulo (Kaggle-style) — usadas só no fim, para gerar a submissão. O treinamento roda inteiro sobre `train`, que tem `Churn`.

In [ ]:
sample_submission = spark.table("workspace.default.churn_sample_submission")
train = spark.table("workspace.default.churn_train")
test = spark.table("workspace.default.churn_test")

## 2. Preparar o alvo e o split dev/teste

`Churn` ("Yes"/"No") vira 0/1. O split usa uma única coluna de sorteio (`_sorteio`) reaproveitada nas duas comparações — chamar `F.rand(seed=42)` de novo em cada `.when()` sorteia números **diferentes** a cada chamada, mesmo com a mesma seed, e quebra a proporção 70/15/15 pretendida (o segundo corte vira independente do primeiro, não condicional).

In [ ]:
from pyspark.sql import functions as F

train = train.withColumn("Churn", F.when(F.col("Churn") == "Yes", 1).otherwise(0))

train = train.withColumn("_sorteio", F.rand(seed=42))
train = train.withColumn(
    "split",
    F.when(F.col("_sorteio") < 0.70, "train")
     .when(F.col("_sorteio") < 0.85, "val")
     .otherwise("test"),
).drop("_sorteio")

train_pd = train.toPandas()
test_pd = test.toPandas()

train_pd["split"].value_counts()

## 3. Iniciar a Esteira (`modelagem_lab.Esteira`)

`Esteira.dividir_por_amostra` é o ponto de entrada: divide `train_pd` em dev/teste usando a coluna `split` que já veio do passo anterior, e devolve o objeto que carrega esse estado dali em diante — não precisa mais segurar `df_dev`/`df_teste` como duas variáveis soltas. `coluna_y="Churn"` já renomeia o alvo para `"y"` — nome que todo o resto do funil exige. O bucket `"val"` fica de fora deste split (reservado como holdout extra — ver nota no fim do notebook, a lib ainda não expõe uma função de "aplicar transformação em dado novo sem re-ajustar" para um terceiro conjunto).

In [ ]:
from modelagem_lab import Esteira

esteira = Esteira.dividir_por_amostra(
    df=train_pd,
    coluna_amostra="split",
    valores_dev=["train"],
    valores_teste=["test"],
    coluna_y="Churn",
)

print(f"dev: {len(esteira.df_dev)} linhas | teste: {len(esteira.df_teste)} linhas")

## 4. Construção de variáveis (opcional)

Duas formas de usar:

1. **`esteira.construir_razoes(pares=[...])`** — você lista os pares com sentido de negócio (ex.: `TotalCharges / tenure` ≡ "gasto médio mensal"), nome que você escolhe. Mais interpretável, exige conhecer o schema. Aplica em dev e teste automaticamente (a `Esteira` cuida da simetria pra você).
2. **`construcao.construir_todas_as_razoes`** (módulo direto, sem wrapper na `Esteira` ainda) — gera razão (e diferença) para **todo par ordenado** de colunas numéricas automaticamente, com nome genérico (`"{a}_sobre_{b}"`, `"{a}_menos_{b}"`). Não exige saber o schema, mas cresce quadraticamente (`n*(n-1)` novas colunas) — rode a pré-seleção (célula 8) logo depois pra filtrar o que não prestou. Como não é um passo dedicado da `Esteira`, o estado (`esteira.df_dev`/`esteira.df_teste`) é manipulado diretamente — são atributos públicos, não um objeto opaco.

Uso aqui: opção 2, já que não conheço o schema exato de `churn_train` neste ambiente.

In [ ]:
import pandas as pd
from pipeline_lab import construcao

# Mesma lista de colunas nos dois lados -- garante que dev/teste ganhem
# exatamente as mesmas colunas novas, mesmo que algum dtype numérico
# divirja entre as duas bases por acaso. Excluir "y" é obrigatório: y já é
# numérica (0/1) nesse ponto, e uma razão/diferença envolvendo o próprio
# alvo (ex.: "tenure_sobre_y") é vazamento garantido por construção -- o
# Pedro_Wise aceitaria com KS artificialmente perfeito.
colunas_numericas = [
    c for c in esteira.df_dev.columns if c != "y" and pd.api.types.is_numeric_dtype(esteira.df_dev[c])
]

novas_dev = construcao.construir_todas_as_razoes(
    df=esteira.df_dev,
    colunas=colunas_numericas,
    incluir_diferenca=True,
    epsilon=1e-6,
)
novas_teste = construcao.construir_todas_as_razoes(
    df=esteira.df_teste,
    colunas=colunas_numericas,
    incluir_diferenca=True,
    epsilon=1e-6,
)
esteira.df_dev = pd.concat([esteira.df_dev, novas_dev], axis=1)
esteira.df_teste = pd.concat([esteira.df_teste, novas_teste], axis=1)

print(f"{novas_dev.shape[1]} razões/diferenças geradas automaticamente a partir de {len(colunas_numericas)} colunas numéricas")
esteira.df_dev.shape, esteira.df_teste.shape

## 5. Agregação temporal (`.agregar_temporal`) — não aplicável aqui

Agregação temporal exige um **painel** (várias linhas por chave ao longo do tempo — ex.: atraso mês a mês do mesmo cliente). O dataset de churn aqui é uma linha por cliente (snapshot), não um painel, então esta etapa é pulada. Fica só como referência de assinatura, caso um dia você tenha um painel de uso mensal do cliente para agregar antes do resto do funil:

```python
esteira.agregar_temporal(
    chave="id_cliente",
    coluna_tempo="safra",
    colunas_valor=["uso_mensal"],
    janelas=[3, 6, 12],
)
```

Chamar isso depois de `esteira.categorizar_e_transformar()` levanta `EtapaForaDeOrdemError` de propósito — a `Esteira` não deixa essa etapa rodar fora de ordem.

## 5.5 Remover colunas de identificação (ID)

**Por que só agora, depois da agregação temporal** — não antes dela: se você TIVER um painel de verdade, a agregação temporal precisa da coluna de chave (`chave="id_cliente"` no exemplo acima) inteira e intacta pra agregar corretamente. Removê-la antes quebraria a agregação temporal. Só depois que ela já cumpriu seu papel (uma linha por chave, chave não é mais necessária como identificador de agregação) é que faz sentido tirá-la — ela nunca teve poder preditivo em si, só serviu de índice de agrupamento.

Uma coluna de ID (ex.: `customerID`) tem um valor **diferente por linha** — em `categorizar` ela cai no caminho "categórica" (cada valor vira seu próprio bin, sem discretização), e como os IDs de dev e teste não se repetem, praticamente TODO valor de teste vira "bin ausente da tabela de WOE" (WOE=0). É a causa mais comum do aviso `"N valor(es) com bin ausente da tabela de WOE"` aparecer com um número gigante — ver a nota completa na célula de Categorização. Removendo aqui, logo antes de Interação/Categorização, evita esse ruído sem atrapalhar nada que ainda precisasse da chave.

Assim como a Construção acima, remover colunas é uma operação de pandas genérica, não lógica de domínio do lab — segue manipulando `esteira.df_dev`/`esteira.df_teste` direto.

In [ ]:
# Heurística: nome contém "id" OU todo valor é único (cardinalidade == nº de linhas).
# Confira a lista antes de rodar em produção -- heurística pode errar
# (ex.: uma coluna numérica contínua rara também pode ter cardinalidade == n).
# Se você rodou a agregação temporal acima, a chave (ex.: "id_cliente") já cumpriu seu
# papel de agregação e cai nesta mesma heurística -- é esperado que suma aqui.
colunas_id = [
    c
    for c in esteira.df_dev.columns
    if c != "y" and ("id" in c.lower() or esteira.df_dev[c].nunique(dropna=True) == len(esteira.df_dev))
]
print(f"Colunas removidas por parecerem ID: {colunas_id}")

esteira.df_dev = esteira.df_dev.drop(columns=colunas_id)
esteira.df_teste = esteira.df_teste.drop(columns=[c for c in colunas_id if c in esteira.df_teste.columns])

## 6. Interação — descoberta de regras (`.descobrir_interacoes`)

RuleFit-style: descobre regras tipo "A > x E B > y" via um ensemble de árvores rasas, materializa como coluna 0/1. `esteira.descobrir_interacoes(...)` já aplica em dev e teste e guarda as colunas geradas em `esteira.colunas_geradas["interacao"]`.

In [ ]:
esteira.descobrir_interacoes(
    colunas_categoricas=None,
    profundidade_maxima=2,
    n_arvores=60,
    min_suporte=0.02,
    max_suporte=0.5,
    max_regras=10,
    permitir_cruzamento_entre_bases=True,
    proporcao_variaveis_por_split=None,
    iv_minimo=0.02,
)

colunas_regra = esteira.colunas_geradas.get("interacao", [])
print(f"Regras de interação estáveis encontradas: {len(colunas_regra)}")
colunas_regra

## 7. Categorização + WOE (`.categorizar_e_transformar`)

Sempre a última etapa antes da pré-seleção/treinamento — gera as versões `_woe`/`_log`/`_bin`/etc. de cada candidata. É também o marco que a `Esteira` usa internamente pra travar as etapas anteriores (`construir_razoes`/`agregar_temporal`/`descobrir_interacoes` levantam `EtapaForaDeOrdemError` se chamadas depois desta).

**Sobre o aviso `"N valor(es) com bin ausente da tabela de WOE — atribuído WOE=0"`**: é o `logger.warning` de `transformacao.woe.aplicar_woe` (não um erro) — dispara quando a base de **teste** tem um valor/categoria que a tabela de WOE, ajustada só em **dev**, nunca viu. Comportamento esperado por design anti-leakage (WOE nunca é reajustado em teste); recebe WOE=0 (neutro) em vez de quebrar. Um número **pequeno** é normal (categoria rara que só apareceu em teste por acaso amostral). Um número **gigante** (dezenas de milhares) quase sempre significa uma coluna de **alta cardinalidade** (ID, chave única, texto livre) sendo tratada como categórica — cada valor de teste é "novo" porque nunca se repete. É pra isso que a célula 5.5 acima remove colunas de ID antes de chegar aqui.

In [ ]:
from pipeline_lab import categorizar


def _log_progresso(coluna: str, iv: float) -> None:
    print(f"  {coluna}: IV={iv:.4f}")


esteira.categorizar_e_transformar(
    gerar_transformacoes_potencia=True,
    gerar_bin_ordinal=True,
    ao_processar_coluna=_log_progresso,
)

print(f"woe_dev: {esteira.df_dev.shape} | woe_teste: {esteira.df_teste.shape}")

# Display estruturado: variável, IV, e a régua de interpretação de Siddiqi
# (classificar_iv) -- dá pra rodar display() no Databricks direto nisto.
resumo_iv = (
    pd.Series(esteira.iv_por_variavel, name="iv")
    .sort_values(ascending=False)
    .to_frame()
    .assign(classificacao=lambda d: d["iv"].map(categorizar.classificar_iv))
)
display(resumo_iv)

## 8. Pré-seleção (`.pre_selecionar`)

Filtra variância → IV → correlação, nesta ordem, antes de entregar ao Pedro_Wise. A `Esteira` já fatia `df_dev`/`df_teste` pelas colunas mantidas internamente — não precisa do `woe_dev = woe_dev[[*colunas_mantidas, "y"]]` manual que o estilo funcional exige.

In [ ]:
esteira.pre_selecionar(
    limiar_variancia=1e-6,
    limiar_iv=0.02,
    limiar_correlacao=0.9,
)
resultado_selecao = esteira.resultado_selecao

print(
    f"candidatas: {resultado_selecao['n_inicial']} -> "
    f"{resultado_selecao['n_apos_variancia']} (variância) -> "
    f"{resultado_selecao['n_apos_iv']} (IV) -> "
    f"{resultado_selecao['n_final']} (correlação)"
)

print(f"\nDescartadas por variância quase-nula ({len(resultado_selecao['colunas_descartadas_variancia'])}):")
print(f"  {resultado_selecao['colunas_descartadas_variancia']}")

print(f"\nDescartadas por IV baixo ({len(resultado_selecao['colunas_descartadas_iv'])}):")
print(f"  {resultado_selecao['colunas_descartadas_iv']}")

print(f"\nDescartadas por correlação alta com uma candidata melhor ({len(resultado_selecao['pares_correlacionados_descartados'])}):")
display(pd.DataFrame(resultado_selecao["pares_correlacionados_descartados"], columns=["mantida", "descartada", "correlacao"]))

print(f"\nMantidas ({resultado_selecao['n_final']}):")
print(f"  {resultado_selecao['colunas_mantidas']}")

print(f"\nesteira.df_dev/df_teste já vêm fatiados pelas colunas mantidas: {esteira.df_dev.shape}, {esteira.df_teste.shape}")

## 9. Treinamento — Pedro_Wise (`.treinar`)

A busca stepwise de verdade (forward/backward, níveis 1 a 3). `criterio="teste"` é o default recomendado (evita o viés de aceitar ruído que `criterio="dev"` tem — ver `REFERENCIA.md`). É a etapa **terminal** da `Esteira`: devolve o `ResultadoTreinamento` diretamente (não `self`), mas também fica salvo em `esteira.resultado_treinamento` caso você queira inspecionar depois sem ter guardado o retorno.

`verbose=True` (default) imprime cada passo aceito **ao vivo**, enquanto a busca roda — qual etapa (`forward_simples`, `transformacao_simples`, `backward_simples`, nível 1/2/2.5/3), qual variável entrou/saiu, e o KS resultante naquele passo. `resultado.historico` guarda a mesma lista de eventos pra revisar depois de terminado.

In [ ]:
resultado = esteira.treinar(
    criterio="teste",
    shadow_probing=False,
    forward_simples=True,
    transformacao_simples_nivel1=True,
    backward_simples_nivel1=True,
    min_vars_para_backward=5,
    forward_duplo=True,
    forward_triplo=True,
    transformacao_simples_nivel2=True,
    backward_simples_nivel2=True,
    n_best_duplo=5,
    n_best_triplo_1=3,
    n_best_triplo_2=3,
    nivel3_ativado=False,
    n_best_backward=2,
    profundidade_maxima_nivel3=2,
    p_valor_maximo=None,
    verbose=True,  # imprime cada passo aceito ao vivo, ver célula de markdown acima
)

print(f"\nVariáveis selecionadas: {resultado.variaveis}")
print(f"KS dev:   {resultado.ks_dev:.4f}")
print(f"KS teste: {resultado.ks_teste:.4f}")
print(f"AUC teste: {resultado.auc_teste:.4f}")
print(f"Taxa de evento dev/teste: {resultado.taxa_evento_dev:.2%} / {resultado.taxa_evento_teste:.2%}")

## 10. Resultados

In [ ]:
print("Histórico da busca (um evento por passo aceito, na ordem em que aconteceram):")
display(pd.DataFrame({"passo": range(1, len(resultado.historico) + 1), "evento": resultado.historico}))

print("Coeficientes:")
display(pd.Series(resultado.coeficientes, name="coeficiente").to_frame())

print("Estatísticas (coeficiente / erro padrão / p-valor — diagnóstico, não influencia a seleção):")
display(pd.DataFrame(resultado.estatisticas).T)

print("Tabela de decis (gains/KS table) na base de teste:")
display(pd.DataFrame(resultado.tabela_decis))

## Notas e próximos passos

- **`Esteira` vs. estilo funcional**: este notebook usa `modelagem_lab.Esteira` (builder encadeável, v0.2.0+) — o mesmo funil escrito no estilo funcional puro (`pipeline_lab.divisao.dividir_por_amostra(...)`, cada etapa como função solta, você mesmo encadeia os retornos) está documentado em `python/pipeline_lab/__init__.py` e `REFERENCIA.md`, caso prefira esse controle mais granular.
- **Ordem do funil garantida em runtime**: chamar `construir_razoes`/`agregar_temporal`/`descobrir_interacoes` depois de `categorizar_e_transformar` levanta `EtapaForaDeOrdemError` na hora — não dá pra errar a ordem silenciosamente usando a `Esteira`.
- **Bucket `"val"`**: reservado de propósito, não usado neste notebook. Hoje `categorizar_e_transformar` ajusta e aplica a transformação WOE numa única chamada — não existe ainda uma função separada de "reaplicar as tabelas já ajustadas em dado novo, sem reajustar". Rodar `categorizar_e_transformar(val, val)` recalcularia o WOE usando o `y` do próprio `val` (vazamento). Se quiser avaliar nesse terceiro holdout ou gerar a submissão em cima de `churn_test` (sem rótulo), é preciso expor esse `aplicar_transformacoes(novos_dados, tabelas_do_dev)` na biblioteca antes — posso construir isso a seguir se for útil.
- **`test_pd`** (carregado no passo 1, vindo de `workspace.default.churn_test`) não foi usado neste notebook pelo mesmo motivo acima.
- Para ajustar qualquer hiperparâmetro, edite o valor no parâmetro nomeado correspondente em cada célula — todos já estão explícitos, não há nada escondido em default silencioso.